# QGFD Paper Experiments — single-GPU driver

Runs all three tracks of the QGFD introductory paper end to end and emits
`paper/REPORT.md` with populated tables. Sized for a **free T4 / P100 (~16 GB)**.

| Track | Script | Cost per model |
| --- | --- | --- |
| 1. Zero-shot: perplexity, noise robustness, attention stats, latency | `scripts/review_experiments.py` | ~5–15 min for 3 seeds |
| 2. LoRA fine-tuning A/B (LoRA-only vs LoRA+QGFD) | `scripts/finetune_qgfd.py` | ~20–60 min for 3 seeds |
| 3. Synthetic multi-hop: induction + passkey | `scripts/eval_synthetic.py` | ~3–10 min for 3 seeds |
| 4. Ablation (smallest model only) | inline below | ~10 min |

**Total: a few GPU-hours.** Each track writes its own aggregate JSON, and the
report builder ingests whatever exists — so you can stop after any track and
still get a coherent (partial) report.

**What the headline claim is.** Zero-shot QGFD is expected to be
perplexity-neutral-to-slightly-worse on clean text. The claim is the *robustness
gap*: QGFD's perplexity should degrade **less** under input noise. Every number
carries a t-based 95% CI over seeds, and the paired (within-seed) statistic is the
one to trust.

In [ ]:
# --- Environment -------------------------------------------------------------
import os, subprocess, sys

REPO_URL, REPO_DIR = "https://github.com/rajboopathiking/TorchDire.git", "TorchDire"
if not os.path.isdir("torchdire"):                 # not already inside the repo
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.40", "peft>=0.10", "datasets>=2.18",
                "accelerate", "matplotlib"], check=False)

In [ ]:
# --- GPU and dtype -----------------------------------------------------------
# bf16 needs compute capability >= 8.0. A T4 is sm_75 and a P100 is sm_60, so on
# free-tier hardware this resolves to fp16, NOT bf16. Both arms use the same dtype,
# so it is not a confound — but the paper must report which one actually ran.
import torch

if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    DTYPE = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
    print(f"{p.name} | sm_{p.major}{p.minor} | {p.total_memory / 2**30:.1f} GiB")
else:
    DTYPE = "float32"
    print("No CUDA device — falling back to CPU/float32. Use --quick settings.")
print("torch", torch.__version__, "| dtype for this run:", DTYPE)

## Configuration

`MODELS` are the three ungated SLMs a 16 GB card can hold. `TinyLlama-1.1B` has
real GQA (32 heads / 4 KV) and is the headline model; `Qwen2.5-0.5B` is the
cross-family check (qwen2 adapter rather than llama).

Set `QUICK = True` for a ~5-minute end-to-end rehearsal on the smallest model
before committing to the full run.

In [ ]:
# --- Configuration -----------------------------------------------------------
QUICK = False          # True => tiny rehearsal on one model, minutes not hours

MODELS = [
    "HuggingFaceTB/SmolLM2-135M",              # llama, 135M
    "Qwen/Qwen2.5-0.5B",                       # qwen2, 0.5B — cross-family
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",      # llama, 1.1B, real GQA — headline
]
SEEDS = (0, 1, 2)
RESULTS = "qgfd_paper_results"

RUN_ZEROSHOT, RUN_FINETUNE, RUN_SYNTHETIC, RUN_ABLATION = True, True, True, True

if QUICK:
    MODELS, SEEDS = ["JackFram/llama-160m"], (0, 1)
    RESULTS = "qgfd_quick_results"

os.makedirs(RESULTS, exist_ok=True)
print(f"{len(MODELS)} model(s) x {len(SEEDS)} seed(s) -> {RESULTS}/")
for m in MODELS:
    print("  ", m)

## Track 1 — Zero-shot: perplexity, noise robustness, attention, latency

Both arms run through the *same* adapter; the only difference is whether `p` comes
from `SoftmaxOperator` or `QGFDOperator`. `verify_patch()` raises if the patch
silently no-ops, so a green run means the operator was genuinely reached.

The baseline is **eager materialised softmax**, not SDPA/FlashAttention — QGFD
needs the explicit probability matrix. Read every latency figure that way.

In [ ]:
# --- Track 1 -----------------------------------------------------------------
import traceback
from dataclasses import replace

from scripts.review_experiments import ExperimentConfig, run_all_seeds

zeroshot = {}
if RUN_ZEROSHOT:
    for mid in MODELS:
        cfg = ExperimentConfig(
            model_id=mid, dtype=DTYPE, device="auto",
            diffusion_steps=1, target_alpha=0.05,
            out_dir=f"{RESULTS}/zeroshot/{mid.split('/')[-1]}",
        )
        if QUICK:
            cfg = replace(cfg, ppl_num_texts=8, robustness_num_texts=6,
                          attn_num_texts=2, ppl_max_length=128, ppl_stride=128,
                          text_pool_size=64, latency_seq_len=128, latency_iters=3,
                          latency_warmup=1, gen_max_new_tokens=8)
        try:
            zeroshot[mid] = run_all_seeds(cfg, seeds=SEEDS)
        except Exception:
            traceback.print_exc()
            print(f"!! zero-shot FAILED for {mid} — continuing with the other models")
print("zero-shot done:", list(zeroshot))

## Track 2 — LoRA fine-tuning A/B

Equal budget, identical adapters on `q/k/v/o`, identical seed / data / LR /
schedule / step count. The only difference is the probability operator, so the
comparison isolates QGFD rather than adapter capacity.

Two guards worth knowing about, because both used to fail silently:

* `verify_lora_live()` runs a probe forward+backward and **refuses to report a
  result** unless a `lora_B` actually receives gradient. The adapter aliases
  `q/k/v/o` onto itself, and `nn.Module.named_modules()` de-duplicates shared
  submodules — so PEFT used to inject LoRA into a module `forward()` never called.
* α warmup is driven by a `TrainerCallback`, and `report_alpha()` checks that
  `step_count` actually advanced past `warmup_steps`. Mutating step state inside
  `forward()` diverges on gradient-checkpoint recompute.

If you are tight on VRAM, lower `batch_size` and raise `grad_accum` — the product
is what matters.

In [ ]:
# --- Track 2 -----------------------------------------------------------------
from scripts.finetune_qgfd import FinetuneConfig, apply_quick
from scripts.finetune_qgfd import run_all_seeds as ft_run_all_seeds

finetune = {}
if RUN_FINETUNE:
    for mid in MODELS:
        cfg = FinetuneConfig(
            model_id=mid, dtype=DTYPE, device="auto", backend="operator",
            diffusion_steps=1, target_alpha=0.05, warmup_steps=100,
            max_steps=300, batch_size=2, grad_accum=8, block_size=256,
            learning_rate=2e-4, gradient_checkpointing=True,
            out_dir=f"{RESULTS}/finetune/{mid.split('/')[-1]}",
        )
        if QUICK:
            cfg = apply_quick(cfg)
        try:
            finetune[mid] = ft_run_all_seeds(cfg, seeds=SEEDS)
        except Exception:
            traceback.print_exc()
            print(f"!! fine-tune FAILED for {mid} — continuing with the other models")
print("fine-tune done:", list(finetune))

## Track 3 — Synthetic multi-hop probes

**Induction.** A random word sequence `S` is shown twice; at each position of the
second copy the model must emit whatever followed the same word in the first copy.
That is the canonical two-hop circuit. A fraction of the second copy can be
replaced by unrelated words (`induction_noise_rates`); corrupted positions are
excluded from scoring, so the remaining ones stay well-posed but must route
through a garbled context — the same axis as the headline robustness claim.

**Passkey.** A 5-digit key buried at a controlled depth in filler text, retrieved
at the end. Decoding uses an explicit greedy loop with `use_cache=False`, so the
probe exercises exactly the operator the model was trained with.

`control_acc` scores the **first** copy, where the answer is unpredictable. It is
the chance-level floor: if induction accuracy is not far above it, the row is
uninformative no matter which arm wins.

In [ ]:
# --- Track 3 -----------------------------------------------------------------
from scripts.eval_synthetic import SyntheticConfig
from scripts.eval_synthetic import apply_quick as syn_quick
from scripts.eval_synthetic import run_all_seeds as syn_run_all_seeds

synthetic = {}
if RUN_SYNTHETIC:
    for mid in MODELS:
        cfg = SyntheticConfig(
            model_id=mid, dtype=DTYPE, device="auto", backend="operator",
            diffusion_steps=1, target_alpha=0.05,
            induction_num_examples=64, induction_seq_len=48,
            induction_noise_rates=(0.0, 0.2, 0.4),
            passkey_num_examples=24, passkey_context_tokens=384,
            out_dir=f"{RESULTS}/synthetic/{mid.split('/')[-1]}",
        )
        if QUICK:
            cfg = syn_quick(cfg)
        try:
            synthetic[mid] = syn_run_all_seeds(cfg, seeds=SEEDS, post_lora=False)
        except Exception:
            traceback.print_exc()
            print(f"!! synthetic FAILED for {mid} — continuing with the other models")
print("synthetic done:", list(synthetic))

## Track 4 — Ablation and the α=0 equivalence check

Smallest model only, to save compute. Two separate things happen here.

**The equivalence check is the important one.** Contribution (1) of the paper is
that QGFD at α=0 is *exactly* softmax. That is a falsifiable claim about the
implementation, and it costs one forward pass to test: if `QGFDOperator(α=0)` and
`SoftmaxOperator` do not give bit-identical logits, the drop-in claim is false and
nothing else in the paper should be believed.

**Then the grid:** `T ∈ {1,2}` × `α ∈ {0.02, 0.05}` × `detach_P ∈ {True, False}`,
one seed each, reporting clean perplexity and degradation at the highest noise
rate. This is for *direction* only — one seed cannot separate settings that differ
by ~1%. (`experiments/ablation.py`'s `QGFDAblator` reports ROUGE/BLEU/BERTScore
from hard-coded heuristics and `QGFDProfiler`'s FLOPs/VRAM are analytic estimates;
neither is used here or anywhere in the report.)

In [ ]:
# --- α=0 must be exactly softmax --------------------------------------------
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from torchdire import QGFDOperator, SoftmaxOperator, wrap_model_with_qgfd_operator

_MID = MODELS[0]
_tok = AutoTokenizer.from_pretrained(_MID)
_ids = _tok("Diffusion over the key graph should vanish when alpha is zero.",
            return_tensors="pt")["input_ids"]

_logits = {}
for _name, _op in (("softmax", SoftmaxOperator()),
                   ("qgfd_alpha0", QGFDOperator(diffusion_steps=1, target_alpha=0.0,
                                                warmup_steps=0, detach_P=True,
                                                mode="full", is_causal=True))):
    _m = AutoModelForCausalLM.from_pretrained(_MID, torch_dtype=torch.float32)
    torch.manual_seed(0)
    _m = wrap_model_with_qgfd_operator(_m, _op, verbose=False).eval()
    with torch.no_grad():
        _logits[_name] = _m(input_ids=_ids).logits.clone()
    del _m

_delta = (_logits["softmax"] - _logits["qgfd_alpha0"]).abs().max().item()
print(f"max |logit difference| at alpha=0: {_delta:.3e}")
assert _delta < 1e-5, ("alpha=0 is NOT equivalent to softmax — the drop-in claim "
                       "is false, fix the operator before reporting anything else")
print("PASS: QGFD at alpha=0 reproduces softmax.")

In [ ]:
# --- Ablation grid (one seed, direction only) --------------------------------
import itertools, json

from scripts.review_experiments import run_all

ablation = []
if RUN_ABLATION:
    _base = ExperimentConfig(
        model_id=MODELS[0], dtype=DTYPE, device="auto",
        ppl_num_texts=60, robustness_num_texts=40, attn_num_texts=8,
        latency_seq_len=256, latency_iters=5, gen_max_new_tokens=8,
        seed=0, robustness_seed=0, text_sample_seed=0,
    )
    if QUICK:
        _base = replace(_base, ppl_num_texts=6, robustness_num_texts=4,
                        attn_num_texts=2, ppl_max_length=128, ppl_stride=128,
                        text_pool_size=64, latency_seq_len=128, latency_iters=2)

    for T, alpha, det in itertools.product((1, 2), (0.02, 0.05), (True, False)):
        tag = f"T{T}_a{alpha}_detach{det}"
        cfg = replace(_base, diffusion_steps=T, target_alpha=alpha, detach_P=det,
                      out_dir=f"{RESULTS}/ablation/{tag}")
        try:
            r = run_all(cfg)
            qg = r["arms"]["qgfd"]
            rate = max(float(k) for k in qg["robustness"])
            base = qg["robustness"][min(qg["robustness"], key=float)]
            worst = qg["robustness"][max(qg["robustness"], key=float)]
            ablation.append({"T": T, "alpha": alpha, "detach_P": det,
                             "clean_ppl": qg["clean_ppl"],
                             "noise_rate": rate,
                             "degradation_pct": 100.0 * (worst - base) / base})
        except Exception:
            traceback.print_exc()
            print(f"!! ablation FAILED for {tag}")

    with open(f"{RESULTS}/ablation/ablation.json", "w") as fh:
        json.dump(ablation, fh, indent=2)
    print(f"\n{'T':>2} {'alpha':>6} {'detach_P':>9} {'clean PPL':>11} {'degr %':>9}")
    for row in ablation:
        print(f"{row['T']:>2} {row['alpha']:>6} {str(row['detach_P']):>9} "
              f"{row['clean_ppl']:>11.4f} {row['degradation_pct']:>9.1f}")
    print("\nOne seed per row — read direction, not differences.")

## Compute overhead vs sequence length

Prefill latency and peak VRAM at L ∈ {128, 256, 512}, per arm. QGFD adds a `K·Kᵀ`
GEMM plus one `p·P` product per diffusion step, so overhead should grow with L —
but the dominant cost is structural, not arithmetic: materialising `p` forecloses
fused attention kernels entirely. Against a FlashAttention baseline the gap would
be larger than anything measured here.

In [ ]:
# --- Latency / VRAM vs sequence length ---------------------------------------
from scripts.review_experiments import benchmark_latency, make_model

overhead = []
_mid = MODELS[-1] if not QUICK else MODELS[0]
for _arm in ("softmax", "qgfd"):
    _cfg = ExperimentConfig(model_id=_mid, dtype=DTYPE, device="auto",
                            latency_iters=10 if not QUICK else 2,
                            latency_warmup=3 if not QUICK else 1)
    _tokL, _mL, _dev = make_model(_arm, _cfg)
    for L in ((128, 256, 512) if not QUICK else (128,)):
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        r = benchmark_latency(_mL, _tokL, _dev, replace(_cfg, latency_seq_len=L))
        overhead.append({"arm": _arm, "seq_len": L, "prefill_ms": r["prefill_ms"],
                         "peak_vram_mb": (torch.cuda.max_memory_allocated() / 2**20
                                          if torch.cuda.is_available() else None)})
    del _mL
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"{'arm':>8} {'L':>5} {'prefill ms':>11} {'peak VRAM MB':>13}")
for r in overhead:
    v = f"{r['peak_vram_mb']:.0f}" if r["peak_vram_mb"] else "n/a"
    print(f"{r['arm']:>8} {r['seq_len']:>5} {r['prefill_ms']:>11.2f} {v:>13}")
for L in sorted({r["seq_len"] for r in overhead}):
    s = next(r for r in overhead if r["arm"] == "softmax" and r["seq_len"] == L)
    q = next(r for r in overhead if r["arm"] == "qgfd" and r["seq_len"] == L)
    print(f"L={L}: QGFD is {q['prefill_ms'] / s['prefill_ms']:.2f}x eager softmax")

with open(f"{RESULTS}/overhead.json", "w") as fh:
    json.dump({"model_id": _mid, "dtype": DTYPE, "rows": overhead}, fh, indent=2)

## Build the report

`scripts/build_report.py` walks the results tree, ingests every aggregate it finds,
and writes `paper/REPORT.md`. Tracks that were skipped or failed appear as
"_Not yet run._" rather than as blanks, and under-powered runs (n < 3) are called
out in the Threats to Validity section automatically.

In [ ]:
# --- Report ------------------------------------------------------------------
from scripts.build_report import build_report, discover

found = discover([RESULTS])
for track, entries in found.items():
    for path, agg in entries:
        print(f"  [{track}] {agg['meta']['model_id']} n={agg['meta']['n_seeds']}")

text = build_report(found, "paper/REPORT.md")
print(f"\nWrote paper/REPORT.md ({len(text.splitlines())} lines)")

try:
    from IPython.display import Markdown, display
    display(Markdown(text))
except ImportError:
    print(text)

## Before believing any of this — the checklist

1. **Did the patch actually apply?** Every track calls a verifier that raises on a
   no-op patch. A green run is the evidence; a run that printed a
   `GenericAttentionAdapter` warning is not.
2. **Did LoRA reach the live projections?** Track 2 prints
   `LoRA live: N trainable tensors, M with non-zero grad`. At initialisation
   `M ≈ N/2` is correct — `lora_B` starts at zero, so `lora_A` gets no gradient
   on the first probe. `M = 0` would have raised.
3. **Did α warm up?** The `alpha` block in each fine-tuning result must show
   `step_count ≥ warmup_steps` and `alpha_train_mode == target_alpha`.
4. **Is n ≥ 3?** With n=2 the t-critical value is 12.7 and essentially nothing can
   reach significance. Two seeds is a rehearsal, not a result.
5. **Read the paired column, not the per-arm columns.** Between-seed corpus
   variance is far larger than the effect; only the within-seed difference has the
   resolution to say anything.
6. **α=0 equivalence must pass.** If it does not, the drop-in claim is false and
   the rest of the report is meaningless.